In [ ]:
from nupack import *
import RNA
import math
import itertools

def find_parentheses_pairs(s):
    stack = []  # Stack to keep track of '(' positions
    pairs = {}  # Dictionary to store pairs of indices (key is '(' or ')', value is the paired parenthesis index)
    
    # Iterate through the string with index
    for i, char in enumerate(s):
        if char == '(':
            stack.append(i)  # Push the index of '(' onto the stack
        elif char == ')':
            if stack:  # Ensure stack is not empty
                open_index = stack.pop()  # Pop the index of the matching '('
                pairs[open_index] = i  # Store the pair (open_index, close_index)
                pairs[i] = open_index  # Store the reverse pair (close_index, open_index)
    
    return pairs

def get_outermost_paired_parenthesis(pairs, position):
    # If position is an opening parenthesis, we find its outermost pair
    if position in pairs:
        # Return the paired parenthesis
        return pairs[position]
    else:
        return None  # If no matching parenthesis found for the given position

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

file_paths = [
    'Feature_guide_baseonly_PAM_complete_HT11_TTTV_filtered_unique_reduced_seq_feature.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  # No need to write anything, just open and close the file to delete its content

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('HT1-1_full_guide_sequences_TTTV_filtered.txt', 'r')    # Input crRNA sequences with AsCas12a direct repeat
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('HT1-1_target_sequences_noPAM_TTTV_filtered.txt', 'r')    # Input crRNA sequences with AsCas12a direct repeat
lines = file2.readlines()
target_array = [line.strip() for line in lines]

for i in range (0, len(guide_array)):

    # Initialize struct_prob_array
    struct_prob_array = []

    guide = guide_array[i]
    target = target_array[i]
    
    # Compute the probability for each suboptimal structure using Boltzmann equilibrium probability distribution (ViennaRNA package)
    struct_prob_array_unit = []
    
    spacer = guide[20:40]
    for j in range (0, 20):
    
        if spacer[j] == "A":
            to_be_added = [1, 0, 0, 0]
        elif spacer[j] == "C":
            to_be_added = [0, 1, 0, 0]
        elif spacer[j] == "G":
            to_be_added = [0, 0, 1, 0]
        elif spacer[j] == "U":
            to_be_added = [0, 0, 0, 1]

        struct_prob_array_unit.append(to_be_added)

    # zero padding to supplement to the longest spacer length
    for j in range (0, 10):
        to_be_added = [0, 0, 0, 0]
        struct_prob_array_unit.append(to_be_added)

    target = target[::-1]
    for j in range (0, 20):
    
        if target[j] == "A":
            to_be_added = [1, 0, 0, 0]
        elif target[j] == "C":
            to_be_added = [0, 1, 0, 0]
        elif target[j] == "G":
            to_be_added = [0, 0, 1, 0]
        elif target[j] == "T":
            to_be_added = [0, 0, 0, 1]

        struct_prob_array_unit.append(to_be_added)

    # zero padding to supplement to the longest spacer length
    for j in range (0, 10):
        to_be_added = [0, 0, 0, 0]
        struct_prob_array_unit.append(to_be_added)
    
            
    struct_prob_array_to_append = [struct_prob_array_unit]
    struct_prob_array.append(struct_prob_array_to_append)

    struct_prob_array = list(itertools.chain.from_iterable(struct_prob_array))
        
    # Open a file in write mode
    with open('Feature_guide_baseonly_PAM_complete_HT11_TTTV_filtered_unique_reduced_seq_feature.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_prob_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')

